In [17]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("../.env")

API_KEY = os.getenv("TIINGO_API_KEY")

if API_KEY is None:
    raise ValueError(
        "TIINGO_API_KEY not found. "
        "Create a .env file in the project root."
    )

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

print("Tiingo API key loaded successfully.")

Tiingo API key loaded successfully.


In [18]:
tickers = [
    "SPY", "QQQ", "IWM", "EFA",
    "EEM", "TLT", "GLD", "VNQ"
]

START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Token {API_KEY}"
}

all_prices = {}

for ticker in tickers:

    print(f"Downloading {ticker}...")

    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"

    params = {
        "startDate": START_DATE,
        "endDate": END_DATE
    }

    success = False

    for attempt in range(1, 4):

        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=(15, 120)
            )

            if response.status_code == 200:

                df = pd.DataFrame(response.json())

                if df.empty:
                    raise ValueError(f"No data returned for {ticker}")

                df["date"] = (
                    pd.to_datetime(df["date"])
                    .dt.tz_localize(None)
                )

                df = (
                    df.set_index("date")
                    .sort_index()
                )

                df.to_csv(
                    DATA_DIR / f"{ticker}_daily.csv"
                )

                all_prices[ticker] = df["adjClose"]

                print(
                    f"{ticker}: {len(df)} rows | "
                    f"{df.index.min().date()} → "
                    f"{df.index.max().date()}"
                )

                success = True
                break

            print(
                f"{ticker}: HTTP {response.status_code} "
                f"(attempt {attempt}/3)"
            )

        except requests.exceptions.RequestException as exc:

            print(
                f"{ticker}: network error "
                f"(attempt {attempt}/3): "
                f"{type(exc).__name__}"
            )

        if attempt < 3:
            time.sleep(5)

    if not success:
        raise RuntimeError(
            f"Failed to download {ticker}"
        )

    time.sleep(2)

SPY: 4024 rows | 2010-01-04 → 2025-12-31
QQQ: 4024 rows | 2010-01-04 → 2025-12-31
IWM: 4024 rows | 2010-01-04 → 2025-12-31
EFA: 4024 rows | 2010-01-04 → 2025-12-31
EEM: 4024 rows | 2010-01-04 → 2025-12-31
TLT: 4024 rows | 2010-01-04 → 2025-12-31
GLD: 4024 rows | 2010-01-04 → 2025-12-31
VNQ: 4024 rows | 2010-01-04 → 2025-12-31


In [19]:
prices = (
    pd.DataFrame(all_prices)
    .sort_index()
)

prices.to_csv(
    DATA_DIR / "adjusted_close.csv"
)

print("Shape:", prices.shape)

print("\nMissing values:")
print(prices.isna().sum())

print(
    "\nDate range:",
    prices.index.min(),
    "→",
    prices.index.max()
)

assert prices.shape[1] == 8
assert prices.isna().sum().sum() == 0

print("\nData validation passed.")

Shape: (4024, 8)

Missing values:
SPY    0
QQQ    0
IWM    0
EFA    0
EEM    0
TLT    0
GLD    0
VNQ    0
dtype: int64

Date range: 2010-01-04 00:00:00 → 2025-12-31 00:00:00

Data validation passed.
